<a href="https://github.com/pln-fing-udelar/cursos/blob/master/EscuelaNLP2026/Tutorial_LLMs_EscuelaNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Epentrenamiento de Grandes Modelos de Lenguaje

Segunda Escuela Sudamericana de NLP - 2026

En este tutorial vamos a ver cómo se entrena un Gran Modelo de Lenguaje (Large Language Model - LLM) o, para ser más exactos, un traductor basado en la arquitectura Transfomer... que es algo parecido.

Bienvenid@s.  

---

---
## Configuración inicial

Comenzamos instalando e importando las bibliotecas que utilizaremos a lo largo del notebook.


---

In [ ]:
#PyTorch
!pip install torch
!pip install torcheval
!pip install datasets

#Scikit-learn: la biblioteca más popular para aprendizaje automático del entorno Python
!pip install scikit-learn

# Visualizaciones
!pip install seaborn
!pip install matplotlib

# Veremos después para qué lo necesitamos
!pip install silabeador

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 6.3 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torcheval.metrics as tm

import seaborn as sn
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from datasets import load_dataset


random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


El objetivo de `torch.device` es muy importante: intentamos utilizar una GPU si es posible, porque de esta forma los cálculos van a ser (mucho) más rápidos.


---
## Entrenando un Transformer

En esta sección construiremos y entrenaremos una red neuronal con arquitectura Transformer. La arquitectura Transformer es la que está detrás de todos los Grandes Modelos de Lenguaje, pero no deja de ser una red neuronal (bastante) más compleja que una red feedforward. Queremos ver hoy cómo se construye y se entrena un LLM con una arquitectura Encoder-Decoder, y cómo podemos utilizarlo para resolver una tarea específica.

El objetivo con el que se entrena un LLM, es predecir la distribución de probabilidad de la posible siguiente palabra (o, para ser más exactos, el siguiente _token_) dados los _tokens_ generados hasta el momento. Esto ha tenido un montón de consecuencias.  

Nuestro "Gran" Modelo de Lenguaje tendrá como tokens caracteres, y la salida del Transformer será la traducción de la entrada a un lenguaje típico del Río de la Plata: el [jeringoso](https://es.wikipedia.org/wiki/Jerigonza), en un completo overkill (*), ya que existe un algoritmo determinista que produce la salida dada la entrada. Pero como aquí hacemos Inteligencia Artificial, vamos a construir una red neuronal con arquitectura Transformer que aprenda a traducir a jeringoso a partir de ejemplos "reales" de palabras traducidas.

Notemos que nuestro modelo es un poquito más complicado que un LLM típico, ya que no solamente predice la siguiente palabra, sino que debe también generar la versión traducida, y por lo tanto se parece más a una tarea seq2seq que a un LLM. De todos modos, como veremos, la arquitectura es la misma y el entrenamiento (a efectos prácticos) también.

(*) En Uruguay, y suponemos que también en Argentina, diríamos "matar una hormiga con un trabuco".

## Reglas del jeringoso

Una palabra se traduce a jeringoso aplicando las siguientes reglas:

1. Separar la palabra en sílabas
2. Tomar la última ocurrencia de una vocal en cada sílaba
3. agregar luego de la vocal la letra "p" y repetir la vocal

Por ejemplo, para la palabra "aeropuerto":

1. a - e - ro - puer - to
2. **a** - **e** - r**o** - pu**e**r - t**o**
3. **apa** - **epe** - r**opo** - pu**epe**r - t**opo**

Entonces, nos queda "apaeperopopuepertopo."

## Probando ChatGPT

ChatGPT *sin razonamiento* (GPT-5 Instant) no es bueno traduciendo palabras a jeringoso. Para el ejemplo anterior, pasando las reglas en el prompt, retorna: **aeperopuepertopo**.

**¿Por qué los LLMs podrían tener problemas para resolver esta tarea?**

---

## Generación de datos de aprendizaje

Como esta es una tarea sencilla, podemos construir una función que realice la tarea utilizando una biblioteca que separa en sílabas (silabeador) y aplicando las reglas ya mencionadas. La siguiente función realiza esto:

In [ ]:
import silabeador



NO_ACC = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u"}


# Traduce a jeringoso una palabra, aplicando un algoritmo determinista
def to_jeringoso(word):
  # Obtengo las sílabas
  try:
    sylls = silabeador.syllabify(word)
  except Exception:
    return None
  out = ""

  # Modifica cada sílaba
  for syll in sylls:
    new_syll = ""
    # Recorro la sílaba de atrás para adelante
    for i in range(len(syll)-1, -1, -1):
      vocal = NO_ACC.get(syll[i].lower(), syll[i].lower())
      # Si es la última vocal, le agrego _p_ más la misma vocal
      # Y ya no sigo, porque lo que había antes queda igual
      if vocal in "aeiou":
        new_syll = syll[i] + "p" + vocal + new_syll
        break
      # de lo contrario, dejo como estaba
      else:
        new_syll = syll[i] + new_syll
    out += syll[0:i] + new_syll
  return out

print(to_jeringoso("aeropuerto"))
print(to_jeringoso("camaleón"))
print(to_jeringoso("cuerpo"))

apaeperopopuepertopo
capamapalepeópon
cueperpopo


Utilizaremos un corpus de palabras disponible en NLTK para generar muchos ejemplos de pares _<palabra_español, palabra_jeringoso>_ , y separaremos el conjunto en train y test.

In [ ]:
import nltk
from nltk.corpus import cess_esp

nltk.download('cess_esp')

spanish_words = cess_esp.words()

print("Cantidad de palabras:", len(spanish_words))

spanish_words = list(set(spanish_words))
print("Cantidad de palabras únicas:", len(spanish_words))

[nltk_data] Downloading package cess_esp to /root/nltk_data...
[nltk_data]   Unzipping corpora/cess_esp.zip.


Cantidad de palabras: 192686
Cantidad de palabras únicas: 25464


In [ ]:
print(cess_esp.words())

['El', 'grupo', 'estatal', 'Electricité_de_France', ...]


In [ ]:

print(spanish_words[0:10])

['polímero', 'deshilachados', 'perdone', 'separadas', 'conecta', 'vislumbrar', 'atravesando', 'ruso-español', 'efectuados', 'astros']


In [ ]:
# "Traducir" a jeringoso
words = []
jeringoso_words = []

# Voy a traducir cada una de las palabras del corpus a jeringoso
for word in spanish_words:
  jeringoso_word = to_jeringoso(word)
  if jeringoso_word is not None:
    words.append(word)
    jeringoso_words.append(jeringoso_word)

# Crear DataFrame con Pandas
data = {'word': words, 'jeringoso': jeringoso_words}
df_jeringoso = pd.DataFrame(data)

# Dividir en train y test
train_df, test_df = train_test_split(df_jeringoso, test_size=0.2, random_state=42)

print("DataFrame original:")
print(df_jeringoso.head())
print("\nTrain DataFrame shape:", train_df.shape)
print("Test DataFrame shape:", test_df.shape)

DataFrame original:
            word                jeringoso
0       polímero         popolípimeperopo
1  deshilachados  depeshipilapachapadopos
2        perdone            peperdoponepe
3      separadas        sepepaparapadapas
4        conecta            coponepectapa

Train DataFrame shape: (20324, 2)
Test DataFrame shape: (5081, 2)


## Tokenizador

Ahora construiremos el tokenizador. Debido a que nuestro problema es a nivel de caracteres, nos vamos a armar un tokenizador de caracteres.

Vamos a agregar tres tokens especiales:

- **\<pad>:** para realizar padding de las secuencias
- **\<bos>:** para marcar el inicio de la secuencia en jeringoso
- **\<eos>:** para marcar el fin de una secuencia

Nuestro tokenizador tiene que contar con dos métodos principales:

- **encode:** dada una secuencia de caracteres (una palabra), retorna la lista de identificadores de los caracteres
- **decode:** dada una lista de identificadores de caracteres, retorna la secuencia de caracteres (la palabra)

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>"]

class CharVocab:
  def __init__(self, chars):
    # Los caracteres son los que me pasan
    chars = sorted(set(chars))
    # Agrego los tokens especiales y obtengo el vocabulario
    self.vocab = SPECIAL_TOKENS + chars
    # Le agrego un identifcador a cada token
    self.v2i = {c:i for i,c in enumerate(self.vocab)}
    # Registro los ids de los caracteres especiales, para no buscarlos cada vez
    self.pad_id = self.v2i["<pad>"]
    self.bos_id = self.v2i["<bos>"]
    self.eos_id = self.v2i["<eos>"]


  # Recibo una secuencia de tokens (i.e. de caracteres)
  # y devuelvo su versión "codificada" con los ids correspondientes
  # Puedo decirle que le agregue un token de EOS o de BOS
  # Que veremos que nos será útil cuando entrenemos
  def encode(self, input, add_eos=False, add_bos=False):
    ids = []
    if add_bos:
      ids.append(self.bos_id)
    for c in input:
      ids.append(self.v2i[c])
    if add_eos:
      ids.append(self.eos_id)
    return ids

  # "Rearmo" la secuencia de tokens a partir de los ids
  def decode(self, ids, show_special_tokens=False):
    chars = []
    for i in ids:
      # Ignora los tokens especiales
      # A menos que me pidan explícitamente que lo sincluya
      if not(show_special_tokens) and i in (self.pad_id, self.bos_id, self.eos_id):
        continue
      chars.append(self.vocab[i])
    return "".join(chars)



Inicializamos el tokenizador con los caracteres de nuestros conjuntos de train y test. Como dijimos antes, nuestros tokens y nuestro vocabulario serán simplemente caracteres.

In [ ]:
# A partir de los pares español, jeringoso
# Actualizo el "vocabulario" de caracteres
# Podemos ver que consideramos al vocabulario de entrada y de salida uno solo
# Lo que en nuestro caso tiene sentido
def make_vocab(pairs):
  chars = set()
  for s, t in pairs:
    chars.update(list(s))
    chars.update(list(t))
  vocab = CharVocab(list(chars))
  return vocab

train_pairs = train_df[["word", "jeringoso"]].values.tolist()
val_pairs = test_df[["word", "jeringoso"]].values.tolist()

vocab = make_vocab(train_pairs + val_pairs)

print("Vocab size:", len(vocab.vocab))

hola_e=vocab.encode("Hola", add_bos=False, add_eos=True)
print("'Hola' ->", hola_e)
print("Decode:", vocab.decode(hola_e))
print("Decode:", vocab.decode(hola_e, show_special_tokens=True))



Vocab size: 93
'Hola' -> [33, 67, 64, 53, 2]
Decode: Hola
Decode: Hola<eos>


## Dataset

Una vez tenemos el tokenizador, podemos construir nuestro Dataset, heredando de la clase abstracta Dataset de PyTorch. El dataset contendrá las parejas del entrenamiento, y el vocabulario.

El método `__getitem__` retorna tres valores que necesitaremos para el entrenamiento:

- La entrada al encoder del transformer (palabra en español) tokenizada
- La entrada al decoder del transformer (palabra en jeringoso) tokenizada (para _teacher forcing_)
- La salida esperada (palabra en jeringoso) tokenizada

In [ ]:
class JeringosoDataset(Dataset):
  def __init__(self, pairs, vocab):
    self.pairs = pairs
    self.vocab = vocab

  def __len__(self):
    return len(self.pairs)

  def __getitem__(self, idx):
    src, tgt = self.pairs[idx]
    src_ids = self.vocab.encode(src, add_eos=True, add_bos=False) # src: ... <eos>
    tgt_in  = self.vocab.encode(tgt, add_eos=False, add_bos=True) # tgt_in: <bos> ...
    tgt_out = self.vocab.encode(tgt, add_eos=True, add_bos=False) # tgt_out: ... <eos>
    return torch.tensor(src_ids, device=device), torch.tensor(tgt_in, device=device), torch.tensor(tgt_out, device=device)

Inicializamos el dataset para train y test:

In [ ]:
train_ds = JeringosoDataset(train_pairs, vocab)
val_ds = JeringosoDataset(val_pairs, vocab)

print("Train dataset size:", len(train_ds))
print("Val dataset size:", len(val_ds))

Train dataset size: 20324
Val dataset size: 5081


In [ ]:
train_ds[0]

(tensor([53, 55, 57, 64, 57, 70, 53, 55, 61, 90, 66,  2], device='cuda:0'),
 tensor([ 1, 53, 68, 53, 55, 57, 68, 57, 64, 57, 68, 57, 70, 53, 68, 53, 55, 61,
         90, 68, 67, 66], device='cuda:0'),
 tensor([53, 68, 53, 55, 57, 68, 57, 64, 57, 68, 57, 70, 53, 68, 53, 55, 61, 90,
         68, 67, 66,  2], device='cuda:0'))

In [ ]:
print("Entrada:",vocab.decode(train_ds[0][0],show_special_tokens=True))
print("Entrada decoder:",vocab.decode(train_ds[0][1],show_special_tokens=True))
print("Salida esperada:", vocab.decode(train_ds[0][2],show_special_tokens=True))

Entrada: aceleración<eos>
Entrada decoder: <bos>apacepeleperapaciópon
Salida esperada: apacepeleperapaciópon<eos>


Para poder entrenar, vamos a tener que crear minibatches (conjuntos de ejemplos). Para eso utilizaremos la clase `torch.utils.data.Dataloader`.

Para poder crear un tensor a partir de un minibatch, todas los ejemplos del minibatch deben tener el mismo largo. Para eso utilizaremos el token especial \<pad> que definimos anteriormente.

La función `collate_fn` toma como entrada los ejemplos de un minibatch, y retorna los tensores `srcs`, `tgts_in` y `tgts_out` (de rango 2, matrices), donde:

- `srcs` debe tener las palabras en español tokenizadas, completando con el token \<pad> para que todas tengan el mismo largo
- `tgts_in` debe tener las palabras en jeringoso tokenizadas para _teacher forcing_, completando con el token \<pad> para que todas tengan el mismo largo
- `tgts_out` debe tener las palabras en jeringoso tokenizadas para la salida esperada, completando con el token \<pad> para que todas tengan el mismo largo

Utilizaremos el método [`nn.utils.rnn.pad_sequence`](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html) para realizar el padding de los ejemplos .

In [ ]:
def collate_fn(batch, pad_id):
  # Los batchs tiene una lista con las tuplas de cada ejemplo
  # zip(*batch) traspone la lista y agrupa por campo
  # Ese *batch lo que hace es pasar cada elemento por separado a zip
  srcs, tgts_in, tgts_out = zip(*batch)
  srcs = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=pad_id)
  tgts_in = nn.utils.rnn.pad_sequence(tgts_in, batch_first=True, padding_value=pad_id)
  tgts_out = nn.utils.rnn.pad_sequence(tgts_out, batch_first=True, padding_value=pad_id)
  return srcs, tgts_in, tgts_out

In [ ]:
# Vamos a pasar los ejemplos de a 128
# Cuando vayamos a usarlos para entrenar los vamos a mezclar cada vez
# En la evaluación no lo hacemos porque no tiene sentido

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True,
                      collate_fn=lambda b: collate_fn(b, vocab.pad_id))
val_dl = DataLoader(val_ds, batch_size=128, shuffle=False,
                    collate_fn=lambda b: collate_fn(b, vocab.pad_id))

print("Train batches:", len(train_dl))
print("Val batches:", len(val_dl))


Train batches: 159
Val batches: 40


In [ ]:
# Veamos el primer minibatch del conjunto de train
primer_batch = next(iter(train_dl))

primer_batch

(tensor([[55, 67, 66,  ...,  0,  0,  0],
         [64, 73, 55,  ...,  0,  0,  0],
         [53, 74, 53,  ...,  0,  0,  0],
         ...,
         [55, 67, 67,  ...,  0,  0,  0],
         [61, 66, 57,  ...,  0,  0,  0],
         [56, 73, 57,  ...,  0,  0,  0]], device='cuda:0'),
 tensor([[ 1, 55, 67,  ...,  0,  0,  0],
         [ 1, 64, 73,  ...,  0,  0,  0],
         [ 1, 53, 68,  ...,  0,  0,  0],
         ...,
         [ 1, 55, 67,  ...,  0,  0,  0],
         [ 1, 61, 68,  ...,  0,  0,  0],
         [ 1, 56, 73,  ...,  0,  0,  0]], device='cuda:0'),
 tensor([[55, 67, 68,  ...,  0,  0,  0],
         [64, 73, 68,  ...,  0,  0,  0],
         [53, 68, 53,  ...,  0,  0,  0],
         ...,
         [55, 67, 68,  ...,  0,  0,  0],
         [61, 68, 61,  ...,  0,  0,  0],
         [56, 73, 57,  ...,  0,  0,  0]], device='cuda:0'))

Podemos ver que los minibatches se convierten en tres matrices (tensores de rango 2): una para la entrada del encoder, otra para la entrada del decoder (_teacher forcing_) y otra para comparar con la salida del modelo.

Podemos ver que en cada matriz hay una única fila que no termina con ceros, que se corresponde con la palabra más larga del minibatch. El cero es el identificador del token `<PAD>`, y se utiliza para completar el resto de las secuencias:

In [ ]:
# Cantidad de tokens que no son el token pad, por secuencia
longitudes = (primer_batch[0] != vocab.pad_id).sum(dim=1)
# Índice de la secuencia más larga
idx_mas_larga = longitudes.argmax()

print("Ids de palabra más larga:\n", primer_batch[0][idx_mas_larga])
print("Palabra más larga:", vocab.decode(primer_batch[0][idx_mas_larga]))
print("Ids de la primer palabra:\n", primer_batch[0][0])
print("Primer palabra:", vocab.decode(primer_batch[0][0]))


Ids de palabra más larga:
 tensor([27, 53, 66, 55, 67, 52, 34, 66, 72, 57, 70, 53, 65, 57, 70, 61, 55, 53,
        66, 67, 52, 56, 57, 52, 29, 57, 71, 53, 70, 70, 67, 64, 64, 67,  2],
       device='cuda:0')
Palabra más larga: Banco_Interamericano_de_Desarrollo
Ids de la primer palabra:
 tensor([55, 67, 66, 71, 61, 56, 57, 70, 53, 56, 53, 71,  2,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       device='cuda:0')
Primer palabra: consideradas


## Modelo Transformer

**Implementaremos la red Transformer.** Vamos a querer que tenga:

- Una capa de embeddings estáticos entrenables para el encoder (torch.nn.Embedding)
- Otra capa de embeddings estáticos entrenables para el decoder (torch.nn.Embedding)
- Una capa de embeddings posicionales (por simplicidad, también utilizaremos embeddings entrenables para las posiciones, torch.nn.Embedding)
- Una capa transformer (torch.nn.Transformer), que se compone de:
    - **d_model:** la dimensión de entrada (de los embeddings)
    - **nhead:** la cantidad de cabezales de atención
    - **num_encoder_layers:** la cantidad de capas en el encoder
    - **num_decoder_layers:** la cantidad de capas en el decoder
    - **dim_feedforward:** la dimensión interna de la capa feedforward
    - **dropout:** el porcentaje de dropout
    - **batch_first:** determina si la primera dimensión es la de batch (poner en True)
- Una capa lineal (torch.nn.Linear) que va de la salida del transformer a la salida final, que tiene tamaño len(vocab). Esta capa devuelve un valor para cada posible palabra siguiente.

El **forward** deberá realizar los siguientes pasos:

1. Sumar los embeddings de la entrada al encoder con los embeddings posicionales
2. Sumar los embeddings de la entrada al decoder con los embeddings posicionales
3. Realizar la pasada por el encoder (`self.transformer.encoder`). Usaremos una máscara para no calcular la atención en los tokens de padding (parámetro src_key_padding_mask)
4. Realizar la pasada por el decoder (self.transformer.decoder), con:
    - Máscaras para los tokens de padding (tanto para lo que viene del encoder, parámetro memory_key_padding_mask, como lo que viene del decoder, parámetro tgt_key_padding_mask)
    - Máscara causal (parámetro tgt_mask, asumiremos que nos la pasan en la llamada forward)
5. Pasada por la capa lineal final

In [ ]:
class MiniTransformer(nn.Module):
  def __init__(self, vocab, d_model=192, nhead=4, num_layers=2, d_ff=512, dropout=0.1):
    super().__init__()

    # Capa de embeddings para la entrada
    self.src_embed = nn.Embedding(len(vocab.vocab), d_model, padding_idx=vocab.pad_id)
    # Capa de embeddings para entrada al decoder
    self.tgt_embed = nn.Embedding(len(vocab.vocab), d_model, padding_idx=vocab.pad_id)
    # Capa de embeddings para aprender las posiciones
    self.pos_enc = nn.Embedding(1024, d_model)

    self.transformer = nn.Transformer(
      d_model=d_model,
      nhead=nhead,
      num_encoder_layers=num_layers,
      num_decoder_layers=num_layers,
      dim_feedforward=d_ff,
      dropout=dropout,
      batch_first=True
    )
    self.out_proj = nn.Linear(d_model, len(vocab.vocab))

  def apply_pos_enc(self, emb):
    return emb + self.pos_enc(torch.arange(emb.size(1), device=emb.device))

  def forward(self, src, tgt_in, src_key_padding_mask, tgt_key_padding_mask, tgt_mask):
    # Ajusto con el positional encoding la entrada al encoder
    src_emb = self.apply_pos_enc(self.src_embed(src))
    # Ajusto con el positional encoding la entrada al decoder
    tgt_emb = self.apply_pos_enc(self.tgt_embed(tgt_in))

    # Utilizo el encoder para codificar la entrada
    mem = self.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

    # Utilizo la salida del encoder, más el target ajustado y genero la salida del decoder
    # Esto devuelve un tensor con la forma (batch, seq_len, d_model)
    # Que representa los estados luego de pasar por las capas del decoder, para cada uno de los tokens del target
    out = self.transformer.decoder(
      tgt_emb,
      mem,
      tgt_mask=tgt_mask,
      tgt_key_padding_mask=tgt_key_padding_mask,
      memory_key_padding_mask=src_key_padding_mask
    )
    return self.out_proj(out) # (B, T, V)

In [ ]:
model = MiniTransformer(vocab=vocab).to(device)

print(model)

MiniTransformer(
  (src_embed): Embedding(93, 192, padding_idx=0)
  (tgt_embed): Embedding(93, 192, padding_idx=0)
  (pos_enc): Embedding(1024, 192)
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=192, out_features=192, bias=True)
          )
          (linear1): Linear(in_features=192, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=192, bias=True)
          (norm1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): T

Necesitamos una forma de construir la máscara causal para el decoder. Para ello, usaremos la función torch.triu, que retorna la matriz triangular superior de la matriz pasada por parámetro. El parámetro diagonal=1 excluye la diagonal principal.

En nuestro caso, aplicamos torch.triu sobre una matriz cuyos elementos tienen inicialmente el valor ($-\infty$). Como resultado, obtenemos una matriz con ($-\infty$) por encima de la diagonal principal y ceros en la diagonal y por debajo de ella.

Esta máscara se suma a los logits de atención antes de aplicar softmax. De esta manera, las posiciones correspondientes a tokens futuros reciben un peso de atención igual a cero, impidiendo que cada token pueda acceder a información posterior.


In [ ]:
def generate_square_subsequent_mask(size, device):
  # máscara causal para el decoder
  mask = torch.ones(size, size, device=device) * float('-inf')
  return torch.triu(mask, diagonal=1)

In [ ]:
# Matriz de valores aleatorios simulando pesos de atención
attention_logits = torch.rand(1, 5, 5).to(device)

# Generar máscara
mask = generate_square_subsequent_mask(5, device)

# Aplicar máscara
masked_weights = attention_logits + mask

# Aplicar softmax
attention_weights = F.softmax(masked_weights, dim=-1)

print(attention_weights)

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4425, 0.5575, 0.0000, 0.0000, 0.0000],
         [0.3208, 0.4457, 0.2336, 0.0000, 0.0000],
         [0.1669, 0.2467, 0.2416, 0.3449, 0.0000],
         [0.1362, 0.2817, 0.1585, 0.1696, 0.2540]]], device='cuda:0')


## Entrenamiento

Vamos a entrenar el modelo con los datos generados anteriormente.Debemos especificar la función de pérdida (en este caso, Entropía Cruzada), que calculará la pérdida comparando la predicción de nuestra red con el caracter siguiente en el ejemplo que se esté procesando.

In [ ]:
# Definimos el Optimizador
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

# Definimos la función de pérdida
loss_fn = nn.CrossEntropyLoss(ignore_index=vocab.pad_id)

def run_epoch(dl, train_mode=True):
  if train_mode:
    model.train()
  else:
    model.eval()

  total = 0.0
  # Recorro los ejemplos del DataLoader
  for src, tgt_in, tgt_out in dl:
    src_key_padding_mask = (src == vocab.pad_id)
    tgt_key_padding_mask = (tgt_in == vocab.pad_id)
    tgt_mask = generate_square_subsequent_mask(tgt_in.size(1), device)

    logits = model(src, tgt_in, src_key_padding_mask, tgt_key_padding_mask, tgt_mask)
    loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

    if train_mode:
      opt.zero_grad()
      loss.backward()
      opt.step()

    total += loss.item()
  return total / len(dl)

# Entrenamos durante 10 épocas
for epoch in range(1, 11):
  tr_loss = run_epoch(train_dl, train_mode=True)
  val_loss = run_epoch(val_dl, train_mode=False)
  print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} | val loss {val_loss:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch 01 | train loss 1.2444 | val loss 0.2657
Epoch 02 | train loss 0.2437 | val loss 0.1254
Epoch 03 | train loss 0.1485 | val loss 0.0837
Epoch 04 | train loss 0.1055 | val loss 0.0610
Epoch 05 | train loss 0.0803 | val loss 0.0484
Epoch 06 | train loss 0.0633 | val loss 0.0395
Epoch 07 | train loss 0.0516 | val loss 0.0320
Epoch 08 | train loss 0.0432 | val loss 0.0266
Epoch 09 | train loss 0.0373 | val loss 0.0256
Epoch 10 | train loss 0.0319 | val loss 0.0223


## Evaluación

Finalmente, evaluaremos el rendimiento del modelo sobre el conjunto de test.

Necesitamos un método de decoding para elegir el siguiente caracter según la distribución de probabilidad de salida. Para ello, implementamos la estrategia de greedy decoding.

Observar que:

- Para la etapa de decodificación, no es necesario recomputar la pasada por el encoder, por lo que lo ejecutamos una vez al principio y reutilizamos su salida durante la decodificación
- La generación es autorregresiva hasta que se genera el token \<eos> o se realizen `max_len` iteraciones
- Podríamos usar KV caché para hacer más eficiente la pasada por el decoder (no lo hacemos por simplicidad)

In [ ]:
from tqdm import tqdm

def greedy_decode(model, src_ids, vocab, max_len, device):
  model.eval()

  # memoria encoder
  src_emb = model.apply_pos_enc(model.src_embed(src_ids))
  mem = model.transformer.encoder(src_emb, src_key_padding_mask=None)

  # generación autorregresiva
  # Comenzamos con el caracter de comienzo de oración
  ys = torch.tensor([[vocab.bos_id]], device=device)
  for _ in range(max_len):
    tgt_emb = model.apply_pos_enc(model.tgt_embed(ys))
    tgt_mask = generate_square_subsequent_mask(ys.size(1), device)
    out = model.transformer.decoder(
      tgt_emb,
      mem,
      tgt_mask=tgt_mask,
      memory_key_padding_mask=None
    )


    logits = model.out_proj(out) # (1,T,V)

    next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True) # (1, 1)

    ys = torch.cat([ys, next_token], dim=1) # (1, t+1)
    if next_token.item() == vocab.eos_id:
      break
  return ys.squeeze(0).tolist() # ids

Nos definimos una función para calcular:

- accuracy, que definimos como la proporción de palabras generadas exactamente igual que la salida esperada.
- accuracy a nivel de caracteres, que definimos como la proporción de caracteres en la salida generada que coinciden con los de la salida esperada

In [ ]:
def accuracy_examples(model, ds, vocab, device):
  exact, char_accs = 0, []
  for i in tqdm(range(len(ds))):
    src_ids, _, tgt_out = ds[i]
    src_ids = src_ids.unsqueeze(0).to(device)
    pred_ids = greedy_decode(model, src_ids, vocab, max_len=64, device=device)
    pred = vocab.decode(pred_ids)
    gold = vocab.decode(tgt_out.tolist())
    exact += int(pred == gold)
    # char-level accuracy (simple, longitudes pueden diferir)
    m = max(len(pred), len(gold))
    match = sum(1 for a,b in zip(pred.ljust(m), gold.ljust(m)) if a==b)
    char_accs.append(match / m)
  return exact/len(ds), sum(char_accs)/len(ds)

Finalmente evaluamos y corremos algunos ejemplos.

In [ ]:
# demos
demo = ["aeropuerto", "siempre", "murciélago", "uruguay", "programa", "pelota", "tokenizar", "pilteador", "maniobras"]
print("\nDemos:")
for w in demo:
  src = torch.tensor([vocab.encode(w, add_eos=True)], device=device)
  pred_ids = greedy_decode(model, src, vocab, 64, device)
  print(f"{w} -> {vocab.decode(pred_ids)}   (gold: {to_jeringoso(w)})")


Demos:
aeropuerto -> apaeperopopuepertopo   (gold: apaeperopopuepertopo)
siempre -> siepemprepe   (gold: siepemprepe)
murciélago -> mupurciépelapagopo   (gold: mupurciépelapagopo)
uruguay -> upurupuguapay   (gold: upurupuguapay)
programa -> propograpamapa   (gold: propograpamapa)
pelota -> pepelopotapa   (gold: pepelopotapa)
tokenizar -> topokepenipizapar   (gold: topokepenipizapar)
pilteador -> pipiltepeapadopor   (gold: pipiltepeapadopor)
maniobras -> mapaniopobrapas   (gold: mapanipiopobrapas)


In [ ]:
# evaluación
em, ca = accuracy_examples(model, val_ds, vocab, device=device)
print(f"Val Exact-Match: {em*100:.1f}% | Char-acc: {ca*100:.1f}%")



Veamos los primeros ejemplos del conjunto de test en donde el modelo se equivoca:

In [ ]:
total = 10

for i in range(len(val_ds)):
  src_ids, _, tgt_out = val_ds[i]
  src_ids = src_ids.unsqueeze(0).to(device)
  pred_ids = greedy_decode(model, src_ids, vocab, max_len=64, device=device)
  src = vocab.decode(src_ids.squeeze(0).tolist())
  pred = vocab.decode(pred_ids)
  gold = vocab.decode(tgt_out.tolist())
  if pred != gold:
    print(f"{src} -> {pred}   (gold: {gold})")
    total -= 1
    if total == 0:
      break

Thierry_Durr -> thieperryddupurrr   (gold: thieperrydupurr)
julio_del_2000 -> jupuliopodepel200   (gold: jupuliopodepel2000)
alias -> apalipiapas   (gold: apaliapas)
Ejecutiva_del_OBC -> epejepecuputipivapadepelb   (gold: epejepecuputipivapadepelopobc)
Ruta_de_la_Plata -> ruputapadepelapapapatapa   (gold: ruputapadepelapaplapatapa)
Policía_Técnica_Judicial -> popolipicípiapatépecnipicapadupudiciapal   (gold: popolipicípiapatépecnipicapajupudipiciapal)
Mahuad -> mapahuapad   (gold: mahupuapad)
disminuiría -> dipismipinuipirípiapa   (gold: dipismiupunipirípiapa)
IU-LV-CA -> ipiuplvapa   (gold: iupulvcapa)
632.0000 -> 632000   (gold: 6320000)


## Visualización de los mecanismos de atención

Utilizaremos la biblioteca **BertViz** para explorar los mecanismos de atención del Transformer.

Visualizaremos los pesos de atención de:

- *self-attention* del encoder.
- *self-attention* del decoder.
- *cross-attention* entre el decoder y la representación producida por el encoder.

Las siguientes celdas definen algunas funciones necesarias para extraer las matrices de atención y preparar los datos para BertViz.

In [ ]:
!pip install -q bertviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 8.1 MB/s eta 0:00:00


In [ ]:
from bertviz import head_view, model_view

def tokens_from_ids(ids, vocab):
  """
  Convierte identificadores de tokens a etiquetas para BertViz.
  """
  tokens = []

  for token_id in ids:
    token = vocab.vocab[int(token_id)]

    # Representaciones más visibles para algunos caracteres
    if token == " ":
      token = "␠"
    elif token == "\n":
      token = "↵"

    tokens.append(token)

  return tokens

In [ ]:
def extract_attentions(model, src_ids, tgt_in_ids):
  """
  Ejecuta el modelo y extrae los pesos de atención de cada capa.

  Retorna tres listas:
    - encoder_attention
    - decoder_attention
    - cross_attention

  Cada elemento tiene dimensiones:

      (batch_size, num_heads, query_length, key_length)
  """

  attention_cache = {}
  original_forwards = {}

  was_training = model.training

  def patch_attention(module, key):
    """
    Reemplaza temporalmente el forward de un módulo MultiheadAttention
    para solicitar los pesos separados por cabeza.
    """

    original_forward = module.forward
    original_forwards[module] = original_forward

    def forward_with_weights(*args, **kwargs):
      kwargs["need_weights"] = True
      kwargs["average_attn_weights"] = False

      output, weights = original_forward(*args, **kwargs)

      attention_cache[key] = weights.detach().cpu()

      return output, weights

    module.forward = forward_with_weights

  # Self-attention del encoder
  for layer_index, layer in enumerate(model.transformer.encoder.layers):
    patch_attention(
      layer.self_attn,
      ("encoder", layer_index)
    )

  # Self-attention y cross-attention del decoder
  for layer_index, layer in enumerate(model.transformer.decoder.layers):
    patch_attention(
      layer.self_attn,
      ("decoder", layer_index)
    )

    patch_attention(
      layer.multihead_attn,
      ("cross", layer_index)
    )

  # PyTorch puede utilizar una implementación optimizada que no pasa
  # por los módulos MultiheadAttention. La desactivamos temporalmente.
  fastpath_was_enabled = None

  if hasattr(torch.backends, "mha"):
    fastpath_was_enabled = torch.backends.mha.get_fastpath_enabled()
    torch.backends.mha.set_fastpath_enabled(False)

  try:
    model.eval()

    tgt_mask = generate_square_subsequent_mask(
      tgt_in_ids.size(1),
      tgt_in_ids.device
    )

    with torch.inference_mode():
      _ = model(
        src_ids,
        tgt_in_ids,
        src_key_padding_mask=None,
        tgt_key_padding_mask=None,
        tgt_mask=tgt_mask
      )

  finally:
    # Restaurar los forward originales
    for module, original_forward in original_forwards.items():
      module.forward = original_forward

    # Restaurar la configuración original de PyTorch
    if fastpath_was_enabled is not None:
      torch.backends.mha.set_fastpath_enabled(
        fastpath_was_enabled
      )

    model.train(was_training)

  num_encoder_layers = len(
    model.transformer.encoder.layers
  )

  num_decoder_layers = len(
    model.transformer.decoder.layers
  )

  encoder_attention = [
    attention_cache[("encoder", layer_index)]
    for layer_index in range(num_encoder_layers)
  ]

  decoder_attention = [
    attention_cache[("decoder", layer_index)]
    for layer_index in range(num_decoder_layers)
  ]

  cross_attention = [
    attention_cache[("cross", layer_index)]
    for layer_index in range(num_decoder_layers)
  ]

  return (
    encoder_attention,
    decoder_attention,
    cross_attention
  )

In [ ]:
def prepare_bertviz_example(word):
  """
  Prepara una palabra y sus matrices de atención para BertViz.
  """

  # Entrada del encoder: caracteres + <eos>
  src_list = vocab.encode(
    word,
    add_eos=True,
    add_bos=False
  )

  src_ids = torch.tensor(
    [src_list],
    device=device
  )

  pred_ids = greedy_decode(
    model,
    src_ids,
    vocab,
    max_len=64,
    device=device
  )

  target = vocab.decode(pred_ids)

  # Entrada del decoder: <bos> + caracteres de la salida
  tgt_in_list = vocab.encode(
    target,
    add_bos=True,
    add_eos=False
  )

  tgt_in_ids = torch.tensor(
    [tgt_in_list],
    device=device
  )

  (
    encoder_attention,
    decoder_attention,
    cross_attention
  ) = extract_attentions(
    model,
    src_ids,
    tgt_in_ids
  )

  example = {
    "encoder_tokens": tokens_from_ids(
      src_list,
      vocab
    ),

    "decoder_tokens": tokens_from_ids(
      tgt_in_list,
      vocab
    ),

    "encoder_attention": encoder_attention,
    "decoder_attention": decoder_attention,
    "cross_attention": cross_attention
  }

  print("Entrada:", word)
  print("Salida esperada:", to_jeringoso(word))
  print("Salida generada:", target)

  return example

### Visualización por cabeza



1. Seleccionar `Encoder`, `Decoder` o `Cross` en el menú superior.
2. Activar y desactivar las distintas cabezas.
3. Pasar el mouse sobre un carácter para observar los pesos.

En la visualización aparecen dos columnas de tokens. Los tokens de la izquierda representan las posiciones que realizan la consulta de atención, mientras que los tokens de la derecha representan las posiciones sobre las que se distribuye esa atención.

Cada línea conecta un token de la izquierda con un token de la derecha. La intensidad de la línea representa el peso de atención asignado: cuanto más intensa es la línea, mayor es la atención que una posición le asigna a la otra.

Cada color corresponde a una cabeza de atención diferente. Como las distintas cabezas pueden aprender patrones distintos, es conveniente observarlas por separado, desactivando temporalmente las demás.

Observar que en las capas *cross-attention*, los caracteres de la izquierda pertenecen al decoder y los de la derecha pertenecen al encoder.

In [ ]:
example = prepare_bertviz_example("aeropuerto")

head_view(
  encoder_attention=example["encoder_attention"],
  decoder_attention=example["decoder_attention"],
  cross_attention=example["cross_attention"],

  encoder_tokens=example["encoder_tokens"],
  decoder_tokens=example["decoder_tokens"],

  prettify_tokens=False
)

Entrada: aeropuerto
Salida esperada: apaeperopopuepertopo
Salida generada: apaeperopopuepertopo


<IPython.core.display.Javascript object>

### Visualización global del modelo

La siguiente vista muestra simultáneamente todas las cabezas de una capa.

Es posible seleccionar:

- El tipo de atención.
- La capa del Transformer.
- Una o varias cabezas de atención.

In [ ]:
model_view(
  encoder_attention=example["encoder_attention"],
  decoder_attention=example["decoder_attention"],
  cross_attention=example["cross_attention"],

  encoder_tokens=example["encoder_tokens"],
  decoder_tokens=example["decoder_tokens"],

  prettify_tokens=False,
  display_mode="light"
)

<IPython.core.display.Javascript object>